In [11]:
import os
os.getcwd()
os.chdir('/Users/lokeshsaipureddi/Desktop/HCAI-Project')

In [12]:
from typing import TypedDict, Annotated, Sequence
from langgraph.graph import StateGraph, END
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from sqlalchemy.orm import Session
from IPython.display import display, Image
from db.database import get_db
from fastapi import Depends
import operator

from services.rag_service import search_professors, search_courses, search_both, suggest_courses
from config import settings

In [13]:
class AgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]
    professor_results: list
    course_results: list
    combined_results: list
    suggestion_results: list

In [14]:
def get_llm():
    """Get configured LLM instance"""
    return ChatOpenAI(
        model="gpt-4",  # or "gpt-3.5-turbo" for faster/cheaper
        temperature=0,
        openai_api_key=settings.OPENAI_API_KEY
    )

In [15]:
SYSTEM_PROMPT = """
You are an intelligent academic assistant
You help students find: 
- Professors (with ratings and detailed student reviews)
- Courses (with course descriptions and expanded coverage of topics taught in the course) 

When answering: 
Be conversational and helpful 
Completely ignore all university affiliations in the search results. Do NOT reject information because it comes from a different university.
Always cite specific professor names, ratings, departments, and highlight student reviews when the user asks about professors 
Always cite specific course codes and titles, and focus more on the topics, content coverage, and learning outcomes when the user asks about courses 
If asked about both professors and courses, provide detailed information on both—student reviews for professors and topic coverage for courses If no relevant results are found, suggest that the user refine their query 
Be honest about limitations—don't make up information Available information from search results will be provided to you. 
Always base your responses strictly on that information.
"""


def router_node(state: AgentState) -> str:
    """
    Use LLM to intelligently decide which search tool(s) to use
    Returns: "search_professors", "search_courses", "search_both",
             "course_suggestions", or "llm"
    """
    llm = get_llm()
    last_message = state["messages"][-1].content

    # Router prompt for the LLM
    router_prompt = f"""You are a query classifier for an academic assistant system.
Your job is to determine what information the user is asking about.

Available options:
1. "professor" - Questions about professors, instructors, faculty, their ratings, teaching style, or reviews
2. "course" - Questions specifically requesting details about a particular course (content, difficulty, credits, syllabus)
3. "both" - Questions involving both professor AND course (e.g., "Who teaches Machine Learning?")
4. "suggestion" - Questions asking for course recommendations, best courses, which course to take, or course advice
5. "general" - General help, greetings, system usage questions

User Query: "{last_message}"

Classify this query. Respond with ONLY ONE WORD:
professor, course, both, suggestion, or general.

Your classification:"""

    # Get LLM classification
    response = llm.invoke([{"role": "user", "content": router_prompt}])
    classification = response.content.strip().lower()

    # Map classification to node names
    routing_map = {
        "professor": "search_professors",
        "course": "search_courses",
        "both": "search_both",
        "suggestion": "course_suggestions",
        "general": "llm"
    }

    # Default to search_both if classification is unclear
    return routing_map.get(classification, "search_both")



def search_professors_node(state: AgentState) -> dict:
    """Search for professors"""
    query = state["messages"][-1].content
    results = search_professors(query)
    return {"professor_results": results}


def search_courses_node(state: AgentState) -> dict:
    """Search for courses"""
    query = state["messages"][-1].content
    results = search_courses(query)
    return {"course_results": results}


def search_both_node(state: AgentState) -> dict:
    """Search both professors and courses"""
    query = state["messages"][-1].content
    results = search_both(query)
    return {"combined_results": results}


def course_suggestions_node(state: AgentState, db:Session, limit = 5) -> dict:
    query = state["messages"][-1].content
    results = suggest_courses(query, limit, db)
    return {"combined_results": results}

In [16]:
def llm_node(state: AgentState) -> dict:
    """Generate response using LLM with search results context"""
    llm = get_llm()

    messages = [SystemMessage(content=SYSTEM_PROMPT)]

    # Add professor results context
    if state.get("professor_results"):
        data = state["professor_results"] 
        prof_context = "**Professor Information Retrieved:**\n"
        prof_context += f"Query: {data.get('query', 'N/A')}\n\n"
        prof_context += f"Summary:\n{data.get('summary', 'No summary available')}\n\n"
        if data.get("sources"):
            prof_context += "📚 Sources used:\n"
            for i, src in enumerate(data["sources"], 1):
                prof_context += f"{i}. {src.get('title', 'Untitled')}\n"
                prof_context += f"   {src.get('snippet', 'No description')}\n\n"
        messages.append(SystemMessage(content=prof_context))


    # Add course results context (dictionary-based response)
    if state.get("course_results"):
        data = state["course_results"] 
        course_context = "**Course Information Retrieved:**\n"
        course_context += f"Query: {data.get('query', 'N/A')}\n\n"
        course_context += f"Summary:\n{data.get('summary', 'No summary available')}\n\n"
        if data.get("sources"):
            course_context += "📚 Sources used:\n"
            for i, src in enumerate(data["sources"], 1):
                course_context += f"{i}. {src.get('title', 'Untitled')}\n"
                course_context += f"   {src.get('snippet', 'No description')}\n\n"
        messages.append(SystemMessage(content=course_context))


    # Add combined results (professor + course info)
    if state.get("combined_results"):
        data = state["combined_results"] 
        combined_context = "**Instructor and Course Information:**\n"
        combined_context += f"Query: {data.get('query', 'N/A')}\n\n"
        combined_context += f"Summary:\n{data.get('summary', 'No summary available')}\n\n"
        if data.get("sources"):
            combined_context += "📘 Sources:\n"
            for i, src in enumerate(data["sources"], 1):
                combined_context += f"{i}. {src.get('title', 'Untitled')}\n"
                combined_context += f"   {src.get('snippet', 'No description')}\n\n"
        messages.append(SystemMessage(content=combined_context))


    # 🔥 NEW: Add course suggestions (RAG-based)
    if state.get("suggestion_results"):
        sugg_context = "**Recommended Courses Based on Your Interest:**\n"
        for i, r in enumerate(state["suggestion_results"], 1):
            data = r.data
            sugg_context += f"{i}. {data.get('course_code', 'N/A')}: {data.get('title', 'Unknown')}\n"
            sugg_context += f"   Description: {data.get('description', 'N/A')[:200]}...\n"
            sugg_context += f"   Difficulty: {data.get('difficulty', 'N/A')}\n"
            sugg_context += f"   Rating: {data.get('rating', 'N/A')}/5.0\n"
            sugg_context += f"   Similarity Score: {r.similarity:.2f}\n\n"
        messages.append(SystemMessage(content=sugg_context))

    # Add conversation history
    for msg in state["messages"]:
        messages.append(msg)

    # Final LLM response
    response = llm.invoke(messages)
    return {"messages": [AIMessage(content=response.content)]}

In [20]:
def create_agent_graph(db: Session) -> StateGraph:
    """
    Create and compile the LangGraph agent with RAG and web search functionality
    
    Args:
        db: Database session for searching
    
    Returns:
        Compiled agent graph
    """
    graph = StateGraph(AgentState)

    # Add nodes
    graph.add_node("search_professors",
                   lambda state: search_professors_node(state))
    graph.add_node("search_courses",
                   lambda state: search_courses_node(state))
    graph.add_node("search_both", lambda state: search_both_node(state))
    graph.add_node("course_suggestions", lambda state: course_suggestions_node(
        state, db))  # RAG-based
    graph.add_node("llm", llm_node)

    # Router
    def route_query(state: AgentState) -> str:
        return router_node(state)

    # Entry mapping
    graph.set_conditional_entry_point(
        route_query,
        {
            "search_professors": "search_professors",
            "search_courses": "search_courses",
            "search_both": "search_both",
            "course_suggestions": "course_suggestions",
            "llm": "llm",
        }
    )

    # 🚨 REMOVED web_search edges — nodes now directly go to llm
    graph.add_edge("search_professors", "llm")
    graph.add_edge("search_courses", "llm")
    graph.add_edge("search_both", "llm")

    # course_suggestions goes directly to LLM after RAG
    graph.add_edge("course_suggestions", "llm")

    # End after LLM
    graph.add_edge("llm", END)

    return graph.compile()

In [21]:
from IPython.display import display, Image
db = next(get_db())
graph = create_agent_graph(db)
display(graph.get_graph().print_ascii())

                                                       +-----------+                                              
                                                   ....| __start__ |....                                          
                                          ......... ...+-----------+.............                                 
                                 .........     .....         .          ....     .........                        
                        .........           ...             .               ....          .........               
                   .....                 ...                .                   ...                .........      
+--------------------+       +-------------+       +----------------+       +-------------------+           ..... 
| course_suggestions |*      | search_both |       | search_courses |       | search_professors |    .......      
+--------------------+ ******+-------------+****   +----------------+       +---

None

In [22]:
query = "Tell me about the course CS 5800 ?"

initial_state = {
        "messages": [HumanMessage(content=query)],
        "professor_results": [],
        "course_results": [],
        "combined_results": []
    }

# Run agent
result = graph.invoke(initial_state)

In [23]:
for m in result["messages"]:
    m.pretty_print()

================================ Human Message =================================

Tell me about the course CS 5800 ?
================================== Ai Message ==================================

The course CS 5800, titled "Algorithms," is a comprehensive study of the mathematical techniques used for the design and analysis of computer algorithms. The course focuses on algorithmic design paradigms and techniques for analyzing the correctness, time, and space complexity of algorithms. 

The topics covered in this course may include:
- Asymptotic notation
- Recurrences
- Loop invariants
- Hoare triples
- Sorting and searching
- Advanced data structures
- Lower bounds
- Hashing
- Greedy algorithms
- Dynamic programming
- Graph algorithms
- NP-completeness

The course is designed for 4.000 credit hours and is primarily lecture-based. It is suitable for Graduate, CPS - Undergraduate Semester, and Undergraduate students. 

Please note that there are no prerequisites, corequisites, or post